# MOUSE configuration 166: why the correction chain matters

> **Supplementary poster/testing notebook.** Original processing date: **2026-09-15**; MoDaCor version: **1.8.0**. This is not part of the core MOUSE correction example.

This notebook makes a poster-ready comparison from the same zirconia-composite measurement used for the pipeline-impact badges. It compares a plain azimuthal integration of the raw sample counts with the fully corrected signal and maps the detector-resolved local change caused by the chain.

The raw and corrected images use the **same final mask, geometry, $q$ map, and logarithmic bins**. Because raw counts and corrected intensity have incompatible units, one robust global scale factor is removed for display. All remaining differences therefore describe correction-induced changes in shape rather than absolute calibration.

In [ ]:
from pathlib import Path
import json
import logging
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir
from modacor.dataclasses.processing_data import ProcessingData
from modacor.io.hdf.hdf_source import HDFSource
from modacor.io.io_sinks import IoSinks
from modacor.io.io_sources import IoSources
from modacor.runner.pipeline import Pipeline

PROJECT_DIR = locate_example_dir("BAM/MOUSE")
sys.path.insert(0, str(PROJECT_DIR))
from mouse_helpers import discover_measurement_pairs


## Configuration

In [ ]:
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "MOUSE_solids.yaml"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "work" / "supplementary" / "poster_2026" / "figures"
SAMPLE_BATCH = 2
CONFIGURATION = int(os.environ.get("MOUSE_CONFIGURATION", "166"))
OUTPUT_STEM = f"MOUSE_{CONFIGURATION}_correction_eyecatcher"
MIN_PIXELS_PER_BIN = 20
MAP_COLOR_QUANTILE = 0.95
MAP_MIN_LIMIT_PERCENT = 10.0
MAP_MAX_LIMIT_PERCENT = 100.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.png"
SVG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.svg"
MAP_PNG_PATH = OUTPUT_DIR / f"MOUSE_{CONFIGURATION}_correction_effect_map.png"
MAP_SVG_PATH = OUTPUT_DIR / f"MOUSE_{CONFIGURATION}_correction_effect_map.svg"
METRICS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_metrics.json"


## Run the real correction chain

The raw detector image is captured immediately after loading `PD_sample`. The corrected image, final mask, geometry, and bin indices are captured after `IP`, so the comparison uses the production pipeline rather than a parallel reimplementation of the corrections. Plotting and downstream uncertainty-combination steps are not required here.

In [ ]:
pair = next(
    candidate
    for candidate in discover_measurement_pairs(
        DATA_DIR, batch_start=SAMPLE_BATCH, batch_end=SAMPLE_BATCH
    )
    if candidate["configuration"] == CONFIGURATION
)

sources = IoSources()
sources.register_source(HDFSource(source_reference="sample", resource_location=pair["sample"]))
sources.register_source(HDFSource(source_reference="background", resource_location=pair["background"]))
processing_data = ProcessingData()
sinks = IoSinks()
scheduler = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH).create_scheduler()
scheduler.prepare()

raw_image = None
reached_indexing = False
previous_logging_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    while scheduler.is_active() and not reached_indexing:
        for node in scheduler.get_ready():
            node.processing_data = processing_data
            node.io_sources = sources
            node.io_sinks = sinks
            node.execute(processing_data)
            step_id = str(node.step_id)
            if step_id == "PD_sample":
                raw_loaded = np.asarray(processing_data["sample"]["signal"].signal, dtype=float)
                acquisition_axes = tuple(range(max(raw_loaded.ndim - 2, 0)))
                raw_image = (
                    np.mean(raw_loaded, axis=acquisition_axes) if acquisition_axes else raw_loaded.copy()
                )
            scheduler.done(node)
            if step_id == "IP":
                reached_indexing = True
                break
finally:
    logging.disable(previous_logging_disable)

if raw_image is None or not reached_indexing:
    raise RuntimeError("Could not capture the raw image and final indexed corrected data.")

sample = processing_data["sample"]
corrected_image = np.asarray(sample["signal"].signal, dtype=float)
q_map = np.asarray(sample["Q"].signal, dtype=float)
final_mask = np.asarray(sample["mask"].signal, dtype=bool)
pixel_index = np.asarray(sample["pixel_index"].signal, dtype=int)

print(f"Sample: {pair['sample'].name}")
print(f"Background: {pair['background'].name}")
print(f"Image shape: {corrected_image.shape}; masked: {100 * final_mask.mean():.1f}%")


## Form a like-for-like comparison

For the 2D map, the display scale is the median absolute corrected/raw ratio over usable nonzero pixels. The plotted conventional percentage is

$$100\left(\frac{I_\mathrm{corrected}}{s I_\mathrm{raw}}-1\right),$$

where $s$ is that one global scale. Negative corrected values may therefore fall below $-100\%$. The signed color range is chosen from the 95th percentile of the absolute local change, constrained to 10–100%, and its actual limits are printed on the colorbar. Grey pixels are masked or have zero raw counts.

The 1D curves are arithmetic azimuthal means. This matches the configured `IndexedAverager` here because its signal weights are unity.

In [ ]:
scale_valid = (
    ~final_mask
    & np.isfinite(raw_image)
    & np.isfinite(corrected_image)
    & (raw_image > 0)
    & (corrected_image != 0)
)
display_scale = float(np.exp(np.median(
    np.log(np.abs(corrected_image[scale_valid]) / raw_image[scale_valid])
)))

relative_change_map = np.full(raw_image.shape, np.nan, dtype=float)
relative_change_map[scale_valid] = 100.0 * (
    corrected_image[scale_valid] / (display_scale * raw_image[scale_valid]) - 1.0
)

curve_valid = (
    ~final_mask
    & (pixel_index >= 0)
    & np.isfinite(raw_image)
    & np.isfinite(corrected_image)
    & np.isfinite(q_map)
)
indices = pixel_index[curve_valid]
n_bins = int(indices.max()) + 1
counts = np.bincount(indices, minlength=n_bins)
safe_counts = np.maximum(counts, 1)
q_curve = np.bincount(indices, weights=q_map[curve_valid], minlength=n_bins) / safe_counts
raw_curve = np.bincount(indices, weights=raw_image[curve_valid], minlength=n_bins) / safe_counts
corrected_curve = np.bincount(
    indices, weights=corrected_image[curve_valid], minlength=n_bins
) / safe_counts

curve_keep = (
    (counts >= MIN_PIXELS_PER_BIN)
    & np.isfinite(q_curve)
    & (raw_curve > 0)
    & (corrected_curve > 0)
)
q_plot = q_curve[curve_keep]
raw_plot = display_scale * raw_curve[curve_keep]
corrected_plot = corrected_curve[curve_keep]
curve_change_percent = 100.0 * (corrected_plot / raw_plot - 1.0)

finite_map_values = relative_change_map[np.isfinite(relative_change_map)]
map_limit_percent = float(np.clip(
    np.quantile(np.abs(finite_map_values), MAP_COLOR_QUANTILE),
    MAP_MIN_LIMIT_PERCENT, MAP_MAX_LIMIT_PERCENT,
))
summary = {
    "sample": pair["sample"].name,
    "background": pair["background"].name,
    "configuration": CONFIGURATION,
    "global_display_scale": display_scale,
    "map_definition_percent": "100 * (corrected / (global_display_scale * raw) - 1)",
    "map_valid_pixel_count": int(finite_map_values.size),
    "map_display_limit_percent": map_limit_percent,
    "map_quantiles_percent": {
        str(q): float(np.quantile(finite_map_values, q))
        for q in (0.05, 0.25, 0.50, 0.75, 0.95)
    },
    "curve_min_change_percent": float(curve_change_percent.min()),
    "curve_max_change_percent": float(curve_change_percent.max()),
}
METRICS_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(f"Removed global display scale: ×{display_scale:.1f}")
print(
    f"Remaining binned-curve change: {curve_change_percent.min():+.0f}% "
    f"to {curve_change_percent.max():+.0f}%"
)


## Poster-ready composite

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titleweight": "bold",
    "axes.labelcolor": "#20242b",
    "text.color": "#20242b",
    "axes.edgecolor": "#707782",
    "xtick.color": "#505761",
    "ytick.color": "#505761",
})

figure = plt.figure(figsize=(14, 6.2), facecolor="white", layout="constrained")
grid = figure.add_gridspec(1, 2, width_ratios=[1.08, 1.0])

curve_axis = figure.add_subplot(grid[0])
curve_axis.loglog(
    q_plot, raw_plot, color="#8b919b", linewidth=2.5, linestyle=(0, (5, 3)),
    label=f"Raw sample × {display_scale:,.0f}\n(display scale only)", zorder=2,
)
curve_axis.loglog(
    q_plot, corrected_plot, color="#a11b8c", linewidth=3.2,
    label="After full correction chain", zorder=3,
)
curve_axis.fill_between(q_plot, raw_plot, corrected_plot, color="#e76fbd", alpha=0.18, zorder=1)
curve_axis.set_xlabel(r"$q$ (nm$^{-1}$)")
curve_axis.set_ylabel("Intensity (common display scale)")
curve_axis.set_title("Integrated signal: before vs after", loc="left", pad=12, fontsize=17)
curve_axis.grid(which="major", alpha=0.18)
curve_axis.grid(which="minor", alpha=0.06)
curve_axis.legend(frameon=False, loc="lower left", fontsize=11)
curve_axis.text(
    0.02, 0.97, r"Same final mask, $q$ map and bins", transform=curve_axis.transAxes,
    va="top", fontsize=10, color="#5d6470",
)

inset = curve_axis.inset_axes([0.56, 0.08, 0.41, 0.28])
inset.semilogx(q_plot, curve_change_percent, color="#a11b8c", linewidth=1.8)
inset.axhline(0, color="#747b85", linewidth=1)
inset.fill_between(q_plot, 0, curve_change_percent, color="#e76fbd", alpha=0.22)
inset.set_ylabel("change (%)", fontsize=8)
inset.set_xlabel(r"$q$", fontsize=8, labelpad=-1)
inset.tick_params(labelsize=7)
inset.grid(alpha=0.15)
inset.set_title("Curve reshaping", fontsize=9, loc="left", pad=2)

map_axis = figure.add_subplot(grid[1])
change_cmap = plt.get_cmap("RdBu_r").copy()
change_cmap.set_bad("#e7e8eb")
image = map_axis.imshow(
    relative_change_map, origin="lower", cmap=change_cmap,
    norm=TwoSlopeNorm(vmin=-map_limit_percent, vcenter=0, vmax=map_limit_percent),
    interpolation="nearest",
)
map_axis.set_title("Detector-resolved correction effect", loc="left", pad=12, fontsize=17)
map_axis.set_xlabel("detector pixel")
map_axis.set_ylabel("detector pixel")
colorbar = figure.colorbar(image, ax=map_axis, shrink=0.88, pad=0.025, extend="both")
colorbar.set_label("local data change after removing global scale (%)")
map_half_limit = map_limit_percent / 2.0
colorbar.set_ticks([-map_limit_percent, -map_half_limit, 0, map_half_limit, map_limit_percent])
colorbar.set_ticklabels([
    f"≤−{map_limit_percent:.0f}", f"−{map_half_limit:.0f}", "0",
    f"+{map_half_limit:.0f}", f"≥+{map_limit_percent:.0f}",
])
map_axis.text(
    0.02, 0.02, "Grey = masked / zero raw counts", transform=map_axis.transAxes,
    fontsize=9, color="#333333",
    bbox={"boxstyle": "round,pad=.3", "facecolor": "white", "edgecolor": "none", "alpha": 0.82},
)

figure.suptitle("Why the correction chain matters", fontsize=23, fontweight="bold", x=0.02, ha="left")
figure.text(
    0.02, 0.93, f"MOUSE zirconia-composite measurement · configuration {CONFIGURATION}",
    fontsize=12, color="#5d6470",
)
figure.savefig(PNG_PATH, dpi=300, bbox_inches="tight", facecolor="white")
figure.savefig(SVG_PATH, bbox_inches="tight", facecolor="white")

map_figure, map_only_axis = plt.subplots(figsize=(7.2, 6.4), facecolor="white", layout="constrained")
map_only_image = map_only_axis.imshow(
    relative_change_map, origin="lower", cmap=change_cmap,
    norm=TwoSlopeNorm(vmin=-map_limit_percent, vcenter=0, vmax=map_limit_percent),
    interpolation="nearest",
)
map_only_axis.set_title("Corrections reshape the detector image", loc="left", pad=12, fontsize=19)
map_only_axis.set_xlabel("detector pixel")
map_only_axis.set_ylabel("detector pixel")
map_colorbar = map_figure.colorbar(map_only_image, ax=map_only_axis, shrink=0.88, pad=0.025, extend="both")
map_colorbar.set_label("local data change after removing global scale (%)")
map_colorbar.set_ticks([-map_limit_percent, -map_half_limit, 0, map_half_limit, map_limit_percent])
map_colorbar.set_ticklabels([
    f"≤−{map_limit_percent:.0f}", f"−{map_half_limit:.0f}", "0",
    f"+{map_half_limit:.0f}", f"≥+{map_limit_percent:.0f}",
])
map_only_axis.text(
    0.02, 0.02, "Grey = masked / zero raw counts", transform=map_only_axis.transAxes,
    fontsize=9, color="#333333",
    bbox={"boxstyle": "round,pad=.3", "facecolor": "white", "edgecolor": "none", "alpha": 0.82},
)
map_figure.savefig(MAP_PNG_PATH, dpi=300, bbox_inches="tight", facecolor="white")
map_figure.savefig(MAP_SVG_PATH, bbox_inches="tight", facecolor="white")
plt.show()
print(f"PNG: {PNG_PATH}")
print(f"SVG: {SVG_PATH}")
print(f"Standalone map PNG: {MAP_PNG_PATH}")
print(f"Standalone map SVG: {MAP_SVG_PATH}")
print(f"Metrics: {METRICS_PATH}")


## Reading the figure

The near-overlap at low $q$ shows why a simple before/after curve can initially make the corrections look unimportant. The inset and detector map reveal the substantial $q$-dependent and spatial reshaping that remains after the large common normalization factor is removed. The map is a diagnostic visualization, not a correction-factor calibration: negative values include the effect of background subtraction, and grey regions have no defined raw/corrected ratio.